# Fast-Sinkhorn-CUDA: Run All Experiments on Colab

**Steps:**
1. Upload your project zip to Colab
2. Build with CMake + CUDA
3. Run all experiments
4. Generate figures
5. Download results

> Make sure to select **GPU runtime**: Runtime → Change runtime type → T4 GPU

## 0. Check GPU

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Upload and extract project

In [ ]:
# Option A: Upload zip file
from google.colab import files
uploaded = files.upload()  # Upload Fast-Sinkhorn-CUDA.zip

import zipfile, os
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/')
        print(f'Extracted {fname}')

# Find the project directory
PROJECT_DIR = '/content/Fast-Sinkhorn-CUDA'
if not os.path.exists(PROJECT_DIR):
    # Try to find it
    for d in os.listdir('/content/'):
        if 'sinkhorn' in d.lower() or 'fast' in d.lower():
            PROJECT_DIR = f'/content/{d}'
            break
print(f'Project directory: {PROJECT_DIR}')
!ls {PROJECT_DIR}

In [ ]:
# Option B: Clone from GitHub (if you pushed the repo)
# !git clone https://github.com/YOUR_USERNAME/Fast-Sinkhorn-CUDA.git /content/Fast-Sinkhorn-CUDA
# PROJECT_DIR = '/content/Fast-Sinkhorn-CUDA'

## 2. Build the project

In [ ]:
BUILD_DIR = f'{PROJECT_DIR}/build'
!mkdir -p {BUILD_DIR}

# Detect GPU architecture for Colab T4 (compute capability 7.5)
!cd {BUILD_DIR} && cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75 \
    2>&1 | tail -5

!cd {BUILD_DIR} && make -j$(nproc) 2>&1 | tail -20

print('\n=== Build targets ===')
!ls -la {BUILD_DIR}/bench_ours {BUILD_DIR}/scaling_experiment {BUILD_DIR}/ablation_* {BUILD_DIR}/stability_* {BUILD_DIR}/convergence_* 2>/dev/null

## 3. Install Python dependencies

In [ ]:
!pip install -q pot geomloss scipy
# PyTorch is already installed on Colab

## 4. Run experiments

In [ ]:
import os
DATA_DIR = f'{PROJECT_DIR}/experiments/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

In [ ]:
%%time
# 4.1 Baseline: our CUDA solver
print('=== Baseline: Ours (CUDA) ===')
!{BUILD_DIR}/bench_ours

In [ ]:
%%time
# 4.2 Baseline: POT (CPU)
print('=== Baseline: POT (CPU) ===')
!python experiments/baselines/bench_pot.py --output-dir {DATA_DIR}

In [ ]:
%%time
# 4.3 Baseline: GeomLoss (GPU)
print('=== Baseline: GeomLoss (GPU) ===')
!python experiments/baselines/bench_geomloss.py --output-dir {DATA_DIR}

In [ ]:
%%time
# 4.4 Baseline: PyTorch Sinkhorn (GPU)
print('=== Baseline: PyTorch Sinkhorn (GPU) ===')
!python experiments/baselines/bench_pytorch_sinkhorn.py --output-dir {DATA_DIR}

In [ ]:
%%time
# 4.5 Scaling experiment
print('=== Scaling Experiment ===')
!{BUILD_DIR}/scaling_experiment

In [ ]:
%%time
# 4.6 Ablation: warp shuffle vs shared memory
print('=== Ablation: Warp Shuffle ===')
!{BUILD_DIR}/ablation_warp_shuffle

In [ ]:
%%time
# 4.7 Ablation: log-domain vs standard
print('=== Ablation: Log Domain ===')
!{BUILD_DIR}/ablation_log_domain

In [ ]:
%%time
# 4.8 Ablation: block size
print('=== Ablation: Block Size ===')
!{BUILD_DIR}/ablation_block_size

In [ ]:
%%time
# 4.9 Ablation: check interval
print('=== Ablation: Check Interval ===')
!{BUILD_DIR}/ablation_check_interval

In [ ]:
%%time
# 4.10 Stability experiment
print('=== Stability ===')
!{BUILD_DIR}/stability_epsilon

In [ ]:
%%time
# 4.11 Convergence profiling
print('=== Convergence Profile ===')
!{BUILD_DIR}/convergence_profile

In [ ]:
# Check all generated data files
print('=== Generated data files ===')
!ls -lh {DATA_DIR}/*.csv

## 5. Merge ablation results

In [ ]:
import pandas as pd
import glob

ablation_files = glob.glob(f'{DATA_DIR}/ablation_*.csv')
dfs = [pd.read_csv(f) for f in ablation_files]
merged = pd.concat(dfs, ignore_index=True)
merged.to_csv(f'{DATA_DIR}/ablation_results.csv', index=False)
print(f'Merged {len(ablation_files)} ablation files -> ablation_results.csv')
merged.head(20)

## 6. Generate all figures

In [ ]:
PAPER_DIR = f'{PROJECT_DIR}/paper'

!python experiments/baselines/plot_baselines.py --data-dir {DATA_DIR} --output-dir {PAPER_DIR}
!python experiments/scaling/plot_scaling.py --data-dir {DATA_DIR} --output-dir {PAPER_DIR}
!python experiments/ablation/plot_ablation.py --data-dir {DATA_DIR} --output-dir {PAPER_DIR}
!python experiments/stability/plot_stability.py --data-dir {DATA_DIR} --output-dir {PAPER_DIR}
!python experiments/convergence/plot_convergence.py --data-dir {DATA_DIR} --output-dir {PAPER_DIR}
!python experiments/applications/color_transfer.py --output-dir {PAPER_DIR}/figures
!python experiments/applications/point_cloud_matching.py --output-dir {PAPER_DIR}/figures

print('\n=== Generated figures ===')
!ls -lh {PAPER_DIR}/figures/*.pdf
print('\n=== Generated tables ===')
!ls -lh {PAPER_DIR}/tables/*.tex

## 7. Preview figures

In [ ]:
from IPython.display import display, Image
import subprocess

# Convert PDFs to PNG for preview
!apt-get -qq install poppler-utils

for pdf in sorted(glob.glob(f'{PAPER_DIR}/figures/*.pdf')):
    png = pdf.replace('.pdf', '.png')
    subprocess.run(['pdftoppm', '-png', '-r', '200', '-singlefile', pdf, png.replace('.png', '')],
                   capture_output=True)
    if os.path.exists(png):
        print(f'\n--- {os.path.basename(pdf)} ---')
        display(Image(filename=png, width=700))

## 8. Download results

In [ ]:
# Pack all results into a zip
import shutil

results_dir = '/content/experiment_results'
os.makedirs(results_dir, exist_ok=True)

# Copy data, figures, and tables
shutil.copytree(DATA_DIR, f'{results_dir}/data', dirs_exist_ok=True)
shutil.copytree(f'{PAPER_DIR}/figures', f'{results_dir}/figures', dirs_exist_ok=True)
shutil.copytree(f'{PAPER_DIR}/tables', f'{results_dir}/tables', dirs_exist_ok=True)

# Create zip
shutil.make_archive('/content/experiment_results', 'zip', results_dir)
print('Created /content/experiment_results.zip')

# Download
from google.colab import files
files.download('/content/experiment_results.zip')

## 9. Quick data summary

In [ ]:
# Print key numbers for the paper
import pandas as pd

print('=== BASELINE COMPARISON (eps=0.01) ===')
for name in ['baseline_ours', 'baseline_pot_cpu', 'baseline_geomloss', 'baseline_pytorch']:
    f = f'{DATA_DIR}/{name}.csv'
    if os.path.exists(f):
        df = pd.read_csv(f)
        df_eps = df[df['epsilon'] == 0.01]
        print(f'\n{name}:')
        print(df_eps[['n', 'time_ms_mean', 'iterations', 'converged']].to_string(index=False))

# Speedup summary
print('\n=== SPEEDUP vs POT ===')
ours = pd.read_csv(f'{DATA_DIR}/baseline_ours.csv')
pot = pd.read_csv(f'{DATA_DIR}/baseline_pot_cpu.csv')
for eps in [0.01]:
    o = ours[ours['epsilon'] == eps].set_index('n')['time_ms_mean'].astype(float)
    p = pot[pot['epsilon'] == eps].set_index('n')['time_ms_mean'].astype(float)
    speedup = p / o
    print(f'eps={eps}:')
    for n, s in speedup.items():
        print(f'  N={n}: {s:.1f}x')